# Linear regression

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn import linear_model
from sklearn.model_selection import train_test_split

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Import the apartment data

In [ ]:
# Define columns for import
columns = [ 'web-scraper-order',
            'address_raw',
            'rooms',
            'area',
            'luxurious',
            'price',
            'price_per_m2',
            'lat',
            'lon',
            'bfs_number',
            'bfs_name',
            'pop',
            'pop_dens',
            'frg_pct',
            'emp',
            'mean_taxable_income',
            'dist_supermarket']

# Read and select variables
df_orig = pd.read_csv("apartments_data_enriched_cleaned.csv", 
                      sep=";", 
                      encoding='utf-8')[columns]

# Rename variable 'web-scraper-order' to 'apmt_id'
df_orig = df_orig.rename(columns={'web-scraper-order': 'id'})

# Remove missing values
df = df_orig.dropna()
df.head(5)

# Remove duplicates
df = df.drop_duplicates()

# Remove some 'extreme' values
df = df.loc[(df['price'] >= 1000) & 
            (df['price'] <= 5000)]

print(df.shape)
df.head(5)

## Simple linear regression (only one explanatory variable in the model)
For details see: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html

### Create train and test samples (train = 80%, test = 20% of the data)

In [ ]:
# Create train and test samples
X_train, X_test, y_train, y_test = train_test_split(df['area'], 
                                                    df['price'], 
                                                    test_size=0.20, 
                                                    random_state=42)
# Show X_train
print('X_train:')
print(X_train.head(), '\n')

# Show y_train
print('y_train:')
print(y_train.head())

### Fit the simple linear regression model

In [ ]:
# Fit the regression model
slope, intercept, r, p, std_err = stats.linregress(X_train, y_train)

# Print results of the regression model
print('Linear regression result:')
print(f'Intercept with y-axis (alpha):            {intercept:.2f}')
print(f'Slope of regression line (beta):          {slope:.3f}')
print(f'p-value:                                  {p:.4f}')
print(f'R-squared (coefficient of determination): {r**2:.4f}')

### Plot regression line

In [ ]:
# Function to calculate model predictions
def myfunc(x):
    return slope * x + intercept

# Apply myfunc() to x, i.e. make predictions 
mymodel = pd.Series(map(myfunc, X_train))

# Scatterplot with regression line
plt.figure(figsize=(6,4))
plt.scatter(X_train, y_train, s=10, color='green')
plt.plot(X_train, mymodel, color='darkred', linestyle='dashed')
plt.title('Simple Linear Regression')
plt.xlabel('area (m2)')
plt.ylabel('price (CHF)')

plt.show()

### Check model residuals (residuals = observed prices minus predicted prices)

In [ ]:
# Calculate model residuals for train data
residuals = y_train - mymodel

# Check the first residual value in our data set
print(f'1st Predicted price in dataset: {mymodel[0]:.2f}')
print(f'1st Observed price in dataset: {y_train[0]:.2f}')
print(f'1st Residual price in dataset: {residuals[0]:.2f}')

### Plot histogram of residuals

In [ ]:
# Plot histogram of residuals
fig = plt.figure( figsize=(7,4))
n, bins, patches = plt.hist(x=residuals, 
                            bins=25, 
                            color='blue',
                            alpha=0.5
                   )

# Set title and labels
plt.xlabel('residuals', fontsize=10, labelpad=10)
plt.ylabel('frequency', fontsize=10, labelpad=10)
plt.title('Histogram of model residuals', fontsize=12, pad=10)

# Show plot
plt.show()

### Compare the observed prices with the predicted prices

In [ ]:
# Create model predictions for test data
predicted = myfunc(X_test)
predicted.round(1)

# Compare the observed prices with the predicted prices
for i in range(0,10):
    print(f'Observed price: {y_test.iloc[i]:.1f}, Predicted price: {predicted.iloc[i]:.1f}')

## Task 1b: Simple linear regression with price_per_m2 as target and area as explanatory variable

In [ ]:
# Create train and test samples with price_per_m2 as target
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(df['area'], 
                                                             df['price_per_m2'], 
                                                             test_size=0.20, 
                                                             random_state=42)

# Fit the regression model
slope_b, intercept_b, r_b, p_b, std_err_b = stats.linregress(X_train_b, y_train_b)

# Print results
print('Linear regression result (price_per_m2 vs area):')
print(f'Intercept with y-axis (alpha):            {intercept_b:.2f}')
print(f'Slope of regression line (beta):          {slope_b:.3f}')
print(f'p-value:                                  {p_b:.4f}')
print(f'R-squared (coefficient of determination): {r_b**2:.4f}')

# Calculate model predictions and residuals
def myfunc_b(x):
    return slope_b * x + intercept_b

mymodel_b = pd.Series(map(myfunc_b, X_train_b))
residuals_b = y_train_b - mymodel_b

# Plot histogram of residuals
fig = plt.figure(figsize=(7,4))
n, bins, patches = plt.hist(x=residuals_b, 
                            bins=25, 
                            color='blue',
                            alpha=0.5)
plt.xlabel('residuals', fontsize=10, labelpad=10)
plt.ylabel('frequency', fontsize=10, labelpad=10)
plt.title('Histogram of model residuals (price_per_m2 vs area)', fontsize=12, pad=10)
plt.show()

### Interpretation (Task 1b):
The R-squared value of the model with price_per_m2 as target and area as explanatory variable is lower than the original model (price vs area). This is because price_per_m2 has less variance explained by area alone compared to the total price. The residuals appear approximately normally distributed with a slight right skew, indicating the model assumptions are reasonably met.

## Task 1c: Simple linear regression with price_per_m2 as target and rooms as explanatory variable

In [ ]:
# Create train and test samples with price_per_m2 as target and rooms as explanatory variable
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(df['rooms'], 
                                                             df['price_per_m2'], 
                                                             test_size=0.20, 
                                                             random_state=42)

# Fit the regression model
slope_c, intercept_c, r_c, p_c, std_err_c = stats.linregress(X_train_c, y_train_c)

# Print results
print('Linear regression result (price_per_m2 vs rooms):')
print(f'Intercept with y-axis (alpha):            {intercept_c:.2f}')
print(f'Slope of regression line (beta):          {slope_c:.3f}')
print(f'p-value:                                  {p_c:.4f}')
print(f'R-squared (coefficient of determination): {r_c**2:.4f}')

# Calculate model predictions and residuals
def myfunc_c(x):
    return slope_c * x + intercept_c

mymodel_c = pd.Series(map(myfunc_c, X_train_c))
residuals_c = y_train_c - mymodel_c

# Plot histogram of residuals
fig = plt.figure(figsize=(7,4))
n, bins, patches = plt.hist(x=residuals_c, 
                            bins=25, 
                            color='blue',
                            alpha=0.5)
plt.xlabel('residuals', fontsize=10, labelpad=10)
plt.ylabel('frequency', fontsize=10, labelpad=10)
plt.title('Histogram of model residuals (price_per_m2 vs rooms)', fontsize=12, pad=10)
plt.show()

### Interpretation (Task 1c):
The R-squared value of the model with price_per_m2 as target and rooms as explanatory variable is significantly lower than the original model (price vs area). This indicates that the number of rooms is a weaker predictor of price per square meter compared to area as a predictor of total price. The residuals show a relatively normal distribution, though with some deviation from perfect normality.

## Multiple linear regression (more than one explanatory variable in the model)
For details see: https://www.statsmodels.org/dev/examples/notebooks/generated/predict.html

### Create train and test samples (train = 80%, test = 20% of the data)

In [ ]:
# Create train and test samples (we name it X2_ and y_2 because we already used X_ and y_ above)
# Task 1d: Adding mean_taxable_income and dist_supermarket as additional variables
X2_train, X2_test, y2_train, y2_test = train_test_split(df[['area',
                                                            'pop_dens',
                                                            'mean_taxable_income',
                                                            'dist_supermarket']], 
                                                        df['price'], 
                                                        test_size=0.20, 
                                                        random_state=42)

# Show X2_train
print('X2_train:')
print(X2_train.head(), '\n')

# Show y2_train
print('y2_train:')
print(y2_train.head())

### Fit the multiple regression model (yes, the output is rich :-), but we need only part of it for interpretation!)

In [ ]:
# Add constant to the model
X2_train_const = sm.add_constant(X2_train)

# Create the multiple regression model
olsmod = sm.OLS(y_train, X2_train_const)
olsres = olsmod.fit()

# Print full model output
print(olsres.summary())

### Interpretation of the relevant (in this course) statistics in the table above

<b>R-squared:</b> This is the coefficient of determination (see slides of lessons). A value of 0.522 means, that the explanatory variables explain 52% of the variaton of our target variable (rental prices) - not bad, but could be improved.

<b>coef:</b> These are the estimated coefficients of the explanatory variables ('slopes of the regression line' of each variable). These are nedded for the price predictions in our model.

<b>P>|t|:</b> These are the p-values. If < 0.05, the explanatory variables shows a statistically siginificant (5% significance level) contribution in explaining the target variable.

<b>Task 1d - Analysis of new variables:</b>
- <b>mean_taxable_income</b>: Check the p-value (P>|t|) for this variable. If it is < 0.05, then this variable is statistically significant at the 5% significance level.
- <b>dist_supermarket</b>: Check the p-value (P>|t|) for this variable. If it is < 0.05, then this variable is statistically significant at the 5% significance level.

### Plot histogram of residuals

In [ ]:
# Plot histogram of residuals
fig = plt.figure( figsize=(8,4))
n, bins, patches = plt.hist(x=olsres.resid, 
                            bins=25, 
                            color='blue',
                            alpha=0.5
                   )

# Set labels
plt.xlabel('residuals', fontsize=10, labelpad=10)
plt.ylabel('frequency', fontsize=10, labelpad=10)
plt.title('Histogram of model residuals', fontsize=12, pad=10)

plt.show()

### Compare the observed prices with the predicted prices

In [ ]:
# Add constant to X2_test
X2_test_const = sm.add_constant(X2_test)
predicted_new = olsres.predict(X2_test_const)

# Compare the observed prices with the predicted prices
for i in range(0,10):
    print(f'Observed price: {y_test.iloc[i]:.1f}, Predicted price: {predicted_new.iloc[i]:.1f}')

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')